# Circular Waveguide Analysis

This notebook demonstrates a complete circular waveguide analysis workflow:

1. Geometry creation
2. Frequency-domain analysis
3. Model order reduction
4. Comparison with analytical solution

In [ ]:
from utils.visualization import *
from geometry.primitives import RectangularWaveguide, CircularWaveguide
from solvers.frequency_domain import FrequencyDomainSolver
from rom.reduction import ModelOrderReduction
from analytical.circular_waveguide import CWGAnalytical
from ngsolve.webgui import Draw # must import Draw, otherwise may run into problems showing mesh

%matplotlib widget
plt.rcParams['figure.dpi'] = 100

## 1. Define Geometry

Create a circular waveguide with specified dimensions and mesh parameters.

In [ ]:
# Waveguide parameters
radius = 150e-3  # Width: 100 mm
L = 300e-3  # Length: 200 mm
maxh = 0.04  # Mesh size

# Create waveguide geometry
cwg = CircularWaveguide(radius, L, maxh=maxh)
# save step file
cwg.save_step(r"./circular_waveguide.step")

print(f"Dimensions: radius={radius * 1e3:.0f}mm, L={cwg.length * 1e3:.0f}mm, L={L * 1e3:.0f}mm")
# print(f"Cutoff frequency (TE10): {cwg.cutoff_frequency_TE11 / 1e9:.3f} GHz")
print(f"Mesh DOFs: ~{cwg.mesh.nv} vertices")

cwg.show('mesh')
cwg.print_info()

## 2. Analytical Solution

Compute the analytical Z-parameters for comparison with numerical results.

In [ ]:
# 2. Analytical
analytical = CWGAnalytical(radius=radius, length=L)

# 3. FOM solve
%time fds = FrequencyDomainSolver(cwg, order=3)
%time fds.assemble_matrices(nportmodes=3)
%time fds.solve(0.1, 1.5, 30, store_snapshots=True)

# 4. Quick comparison plots - just pass the objects!
frequencies = np.linspace(0.1, 1.5, 1000) * 1e9
plot_z_comparison([analytical, fds], frequencies=frequencies, mode_pairs=[(3, 3), (3, 3)])
plot_s_comparison([analytical, fds], frequencies=frequencies, mode_pairs=[(3, 3), (3, 3)])

In [ ]:
fds.port_solver.print_info()

In [ ]:
fds.plot_port_mode('port1', 1)

In [ ]:
fds.plot_port_mode('port2', 1)

In [ ]:
fds.plot_field(14)

In [ ]:
# # 6. Eigenfrequency comparison
# fig, ax = plot_eigenfrequencies([fds], analytical=analytical, n_modes=50)
# ax.set_ylim(0.1, 5)
# ax.set_xlim(0.1, 5)

In [ ]:
# After ROM is working...
from rom.reduction import ModelOrderReduction

rom = ModelOrderReduction(fds)
rom.reduce(max_rank=50, tol=1e-6)
rom.solve(0.1, 1.5, 1000)

# Compare all three
plot_z_comparison([analytical, fds, rom], frequencies=frequencies,
                  labels=['Analytical', 'FOM', 'ROM'])
plot_s_comparison([analytical, fds, rom], frequencies=frequencies,
                  labels=['Analytical', 'FOM', 'ROM'])


In [ ]:
# fig, ax = plot_eigenfrequencies([fds, rom], analytical=analytical,
#                       labels=['FOM', 'ROM'], n_modes=50)
# ax.set_ylim(1, 5)
# ax.set_xlim(1, 5)

In [ ]:
# Compare all three
plot_z_comparison([analytical, fds, rom], frequencies=frequencies,
                  labels=['Analytical', 'FOM', 'ROM'], mode_pairs=[(3, 3), (3, 3)])
plot_s_comparison([analytical, fds, rom], frequencies=frequencies,
                  labels=['Analytical', 'FOM', 'ROM'], mode_pairs=[(3, 3), (3, 3)])

In [ ]:
fig, ax = plot_eigenfrequencies([fds, rom], analytical=analytical,
                      labels=['FOM', 'ROM'], n_modes=50)
ax.set_ylim(0.1, 2)
ax.set_xlim(0.1, 2)
plt.show()

In [ ]:
# Eigenvalue access
eigs = fds.eigenvalues  # Dict with 'global', domain keys
freqs = fds.get_resonant_frequencies(n_modes=20)
fds.print_eigenfrequencies(n_modes=30)

# Eigenmode visualization
fds.plot_eigenmode(mode_index=0, component='abs')
fds.plot_eigenmodes(n_modes=5)

In [ ]:
# Same interface!
eigs = rom.eigenvalues
freqs = rom.get_resonant_frequencies(n_modes=20)
rom.plot_eigenmode(mode_index=0)

In [ ]:
# Same interface!
eigs = concat.eigenvalues
freqs = concat.get_resonant_frequencies(n_modes=20)
concat.plot_eigenmode(mode_index=0)